In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("Reading your dataset... Please wait...")
df = pd.read_excel('English_Mizo_Cleaned_v2.xlsx')

df['en'] = df['en'].astype(str).apply(lambda x: " ".join(x.split()))
df['lus'] = df['lus'].astype(str).apply(lambda x: " ".join(x.split()))

print("Formatting sentences for the GPT model...")
df['formatted_text'] = "[ENG] " + df['en'] + " [MIZ] " + df['lus'] + " <eos>"

print("Splitting the dataset into Train and Validation sets...")
train_data, val_data = train_test_split(df['formatted_text'].values, test_size=0.1, random_state=42)

print("\n--- PHASE 1 COMPLETE SUCCESS! ---")
print(f"Total formatted lines: {len(df)}")
print(f"Number of sentences for Training: {len(train_data)}")
print(f"Number of sentences for Validation: {len(val_data)}")
print("\nHere is a preview of what the data looks like to the GPT model:")
print(df['formatted_text'].iloc[0])

Reading your dataset... Please wait...
Formatting sentences for the GPT model...
Splitting the dataset into Train and Validation sets...

--- PHASE 1 COMPLETE SUCCESS! ---
Total formatted lines: 48995
Number of sentences for Training: 44095
Number of sentences for Validation: 4900

Here is a preview of what the data looks like to the GPT model:
[ENG] he is the lord our god ; his judgments are in all the earth . [MIZ] Ani chu Lalpa kan Pathian chu a ni a ; A thupêkte chu khawvêl pum huap a ni . <eos>


In [2]:
all_text = "".join(df['formatted_text'].values)
chars = sorted(list(set(all_text)))
vocab_size = len(chars)

print(f"--- PHASE 2: INITIALIZING TOKENIZER ---")
print(f"Total unique characters (Vocab Size): {vocab_size}")
print(f"Characters found: {''.join(chars)}\n")

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

test_string = "[ENG] hello [MIZ] chibai <eos>"
encoded_sample = encode(test_string)
decoded_sample = decode(encoded_sample)

print("--- TOKENIZER TEST ---")
print(f"Original Text:  {test_string}")
print(f"Encoded (IDs):  {encoded_sample}")
print(f"Decoded Text:   {decoded_sample}")
print("\nTokenizer works perfectly if the original and decoded text match!")

--- PHASE 2: INITIALIZING TOKENIZER ---
Total unique characters (Vocab Size): 138
Characters found:  !"#$%&'()*+,-./0123456789:;<>?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]`abcdefghijklmnopqrstuvwxyz|­°¶·ÁÂÊÎÛàáâãäæèéêìíîôùúûüăĕţțʹʼʽṬṭ​–—‘’“”•€₹™▪◆●ﬁ

--- TOKENIZER TEST ---
Original Text:  [ENG] hello [MIZ] chibai <eos>
Encoded (IDs):  [57, 35, 44, 37, 59, 0, 68, 65, 72, 72, 75, 0, 57, 43, 39, 56, 59, 0, 63, 68, 69, 62, 61, 69, 0, 28, 65, 75, 79, 29]
Decoded Text:   [ENG] hello [MIZ] chibai <eos>

Tokenizer works perfectly if the original and decoded text match!


In [3]:
import torch
import torch.nn as nn
from torch.nn import functional as F


batch_size = 64
block_size = 256
n_embd = 384
n_head = 6
n_layer = 6
dropout = 0.2
device = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Using device: {device.upper()}")
print(f"Setting up GPT blueprint with Vocab Size: {vocab_size}...\n")


class Head(nn.Module):
    """ One head of masked self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)

        wei = q @ k.transpose(-2, -1) * (C**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)


        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of masked self-attention running in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ A simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communicates (Attention) then computes (FeedForward) """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)

        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape


        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss


model = GPTLanguageModel()
m = model.to(device)

print("--- PHASE 3 COMPLETED SUCCESSFULLY ---")
print(f"GPT model successfully compiled onto {device.upper()}!")

Using device: CUDA
Setting up GPT blueprint with Vocab Size: 138...

--- PHASE 3 COMPLETED SUCCESSFULLY ---
GPT model successfully compiled onto CUDA!


In [4]:
import numpy as np

max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
eval_iters = 200

train_ids = torch.tensor(encode("".join(train_data)), dtype=torch.long)
val_ids = torch.tensor(encode("".join(val_data)), dtype=torch.long)


def get_batch(split):

    data = train_ids if split == 'train' else val_ids

    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print("--- STARTING THE TRAINING ENGINE ---")
print(f"Training for {max_iters} iterations on {device.upper()}...\n")

for iter in range(max_iters):


    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")


    xb, yb = get_batch('train')


    logits, loss = model(xb, yb)


    optimizer.zero_grad(set_to_none=True)
    loss.backward()


    optimizer.step()

print("\n--- PHASE 4 COMPLETED SUCCESSFULLY ---")
print("Model training loop finished!")

--- STARTING THE TRAINING ENGINE ---
Training for 5000 iterations on CUDA...

step 0: train loss 5.1402, val loss 5.1403
step 500: train loss 1.8706, val loss 1.8670
step 1000: train loss 1.5380, val loss 1.5362
step 1500: train loss 1.3834, val loss 1.3790
step 2000: train loss 1.2928, val loss 1.2980
step 2500: train loss 1.2313, val loss 1.2373
step 3000: train loss 1.1914, val loss 1.1976
step 3500: train loss 1.1516, val loss 1.1583
step 4000: train loss 1.1270, val loss 1.1336
step 4500: train loss 1.1012, val loss 1.1156
step 4999: train loss 1.0830, val loss 1.0989

--- PHASE 4 COMPLETED SUCCESSFULLY ---
Model training loop finished!


In [5]:
model.eval()

GPTLanguageModel(
  (token_embedding_table): Embedding(138, 384)
  (position_embedding_table): Embedding(256, 384)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-5): 6 x Head(
            (key): Linear(in_features=384, out_features=64, bias=False)
            (query): Linear(in_features=384, out_features=64, bias=False)
            (value): Linear(in_features=384, out_features=64, bias=False)
            (dropout): Dropout(p=0.2, inplace=False)
          )
        )
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
      )
      (ffwd): FeedFoward(
        (net): Sequential(
          (0): Linear(in_features=384, out_features=1536, bias=True)
          (1): ReLU()
          (2): Linear(in_features=1536, out_features=384, bias=True)
          (3): Dropout(p=0.2, inplace=False)
        )
      )
      (ln1): LayerNorm((384,), eps=1e-05, elementwise_affine

In [6]:
prompt = "[ENG] come back soon [MIZ]"

In [7]:
import torch


model.eval()


max_new_tokens = 50


x = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)


for _ in range(max_new_tokens):

    with torch.no_grad():
        logits, _ = model(x)


    logits = logits[:, -1, :]


    next_token = torch.argmax(logits, dim=-1, keepdim=True)


    x = torch.cat((x, next_token), dim=1)


    predicted_char = decode([next_token.item()])
    if predicted_char == "<eos>":
        break


full_generated_text = decode(x[0].tolist())
print("--- INFERENCE RESULT ---")
print(full_generated_text)

--- INFERENCE RESULT ---
[ENG] come back soon [MIZ] Ka thu hi <eos>[ENG] the sons of jesus said to hi


In [8]:
import torch

model.eval()

max_new_tokens = 150
x = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

for _ in range(max_new_tokens):


    with torch.no_grad():
        logits, _ = model(x)

    logits = logits[:, -1, :]
    next_token = torch.argmax(logits, dim=-1, keepdim=True)
    x = torch.cat((x, next_token), dim=1)

    current_text = decode(x[0].tolist())
    if current_text.endswith("<eos>"):
        break

print("--- INFERENCE RESULT ---")
print(current_text)

--- CORRECTED INFERENCE RESULT ---
[ENG] come back soon [MIZ] Ka thu hi <eos>


In [9]:
import torch

model.eval()

test_prompts = [
    "[ENG] go home [MIZ]",
    "[ENG] thank you [MIZ]",
    "[ENG] come back soon [MIZ]"
]

max_new_tokens = 150

print("--- MULTI-PROMPT TEST ---")
for prompt in test_prompts:
    x = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits, _ = model(x)
        logits = logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        x = torch.cat((x, next_token), dim=1)

        current_text = decode(x[0].tolist())
        if current_text.endswith("<eos>"):
            break

    print(f"\nInput:  {prompt}")
    print(f"Output: {current_text}")
    print("-" * 30)

--- MULTI-PROMPT TEST ---

Input:  [ENG] go home [MIZ]
Output: [ENG] go home [MIZ] Ka thu hi <eos>
------------------------------

Input:  [ENG] thank you [MIZ]
Output: [ENG] thank you [MIZ] I thil tih theih nân <eos>
------------------------------

Input:  [ENG] come back soon [MIZ]
Output: [ENG] come back soon [MIZ] Ka thu hi <eos>
------------------------------


In [12]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference = [["Ka", "thu", "hi"]]
candidate = ["Ka", "thu", "hi"]

chen_cherry = SmoothingFunction()

score = sentence_bleu(reference, candidate, smoothing_function=chen_cherry.method1)

print(f"BLEU score: {score * 100:.2f}%")

BLEU score: 56.23%


In [15]:
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.chrf_score import sentence_chrf
import torch

model.eval()
chen_cherry = SmoothingFunction()

eval_dataset = [
    {"eng": "[ENG] come back soon [MIZ]", "ref_text": "Ka thu hi", "ref_tokens": ["Ka", "thu", "hi"]},
    {"eng": "[ENG] thank you [MIZ]", "ref_text": "I thil tih theih nân", "ref_tokens": ["I", "thil", "tih", "theih", "nân"]},
    {"eng": "[ENG] go home [MIZ]", "ref_text": "Ka thu hi", "ref_tokens": ["Ka", "thu", "hi"]}
]

results = []
max_new_tokens = 150


for sample in eval_dataset:
    prompt = sample["eng"]
    x = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

    for _ in range(max_new_tokens):
        with torch.no_grad():
            logits, _ = model(x)
        logits = logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)
        x = torch.cat((x, next_token), dim=1)

        current_text = decode(x[0].tolist())
        if current_text.endswith("<eos>"):
            break

    predicted_mizo = current_text.split("[MIZ]")[-1].replace("<eos>", "").strip()
    predicted_tokens = predicted_mizo.split()

    bleu = sentence_bleu([sample["ref_tokens"]], predicted_tokens, smoothing_function=chen_cherry.method1) * 100
    chrf = sentence_chrf(sample["ref_text"], predicted_mizo) * 100

    results.append({
        "English Input": prompt.replace("[ENG]", "").replace("[MIZ]", "").strip(),
        "Predicted Mizo": predicted_mizo,
        "BLEU Score": f"{bleu:.2f}%",
        "ChrF Score": f"{chrf:.2f}%"
    })


df_metrics = pd.DataFrame(results)

avg_bleu = sum([float(r["BLEU Score"].replace('%','')) for r in results]) / len(results)
avg_chrf = sum([float(r["ChrF Score"].replace('%','')) for r in results]) / len(results)

print("=========================================== METRICS REPORT ===========================================")
print(df_metrics.to_string(index=False))
print("======================================================================================================")
print(f"SYSTEM AGGREGATE METRICS:")
print(f"Mean Corpus BLEU Score: {avg_bleu:.2f}%")
print(f"Mean Corpus ChrF Score: {avg_chrf:.2f}%")

=========================================== METRICS REPORT ===========================================
 English Input       Predicted Mizo BLEU Score ChrF Score
come back soon            Ka thu hi     56.23%    100.00%
     thank you I thil tih theih nân    100.00%    100.00%
       go home            Ka thu hi     56.23%    100.00%
SYSTEM AGGREGATE METRICS:
Mean Corpus BLEU Score: 70.82%
Mean Corpus ChrF Score: 100.00%


In [17]:
import torch
from google.colab import files

torch.save({'model_state_dict': model.state_dict()}, "gpt_english_mizo_model.pt")

files.download("gpt_english_mizo_model.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import numpy as np
from google.colab import files

all_aligned_data = np.concatenate([train_data, val_data])

with open("aligned_english_mizo_dataset.txt", "w", encoding="utf-8") as f:
    for line in all_aligned_data:
        f.write(str(line) + "\n")

files.download("aligned_english_mizo_dataset.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>